# Bird CLEF 2026 Japanese Tutorial (日本語チュートリアル)
このノートブックでは、環境音データを調査し、鳥の種類を予測するためのベースラインモデルを実装します。

![Bird CLEF](https://www.kaggle.com/competitions/129329/images/header)

## コンペティション概要
BirdCLEFは、Kaggle上で開催される**生物音響（bioacoustics）×機械学習**の国際コンペであり、<br>
自然環境で録音された音声から鳥類の種を識別するAIモデルを開発することが目的です。<br>
具体的には、**野外に設置された録音機器（Passive Acoustic Monitoring）や長時間の環境音(soundscape)** から鳥の鳴き声を検出し、
どの種の鳥かを分類するモデルを作成します。

この種のタスクは、生態学・環境科学・保全科学において重要な問題です。<br>
機械学習により、研究者が大量の音声データを自動解析できるようになります。<br>

---

### 課題設定・目的

森林の音や昆虫の音、複数の鳥が同時に鳴く音といったノイズの多い環境音データ(soundspase)から各音声区間に対して鳥の種類(species)を予測する。つまり、多ラベル音声分類(multi-label audio classification)です。コンペの目的としては以下の3点が存在します。

1. 「鳥類が環境変化や森林破壊、気候変動の影響を敏感に受ける」と言う性質を利用し、鳥類の分布から生態系の健康状態を評価すること
2. 大量の音声データを精密に処理することができるAIモデルを開発することによって従来の専門家による手動解析や種同定の時間や手間を省くこと
3. 十分に研究されていない種やデータの少ない種の鳥類音声を対象とすることで小データ環境での識別方法を開拓すること

参考: [Improving learning-based birdsong classification by utilizing combined audio augmentation strategies](https://www.sciencedirect.com/science/article/pii/S1574954124002413?utm_source=chatgpt.com)

---

### モチベーション・チャレンジ点
モチベーション
1. 生物多様性モニタリングの自動化
世界各地で設置されている自動録音装置により大量の自然音データが収集されているが、専門家による手作業での分析は非常に時間がかかる。
機械学習によって鳥類の鳴き声を自動識別することで、生態系の監視や保全研究を効率化することが期待される。

2. 環境変化の指標としての鳥類
鳥は気候変動や生息環境の変化に敏感な生物であり、その分布や活動を追跡することは環境状態の重要な指標となる。

3. 音響データ解析技術の発展
大規模な自然音データを扱うこの課題は、音声認識、生物音響学、機械学習の交差領域における技術発展を促す。

チャレンジ点
1. 強い環境ノイズ
    - 録音には風・雨・昆虫・人間活動などの雑音が含まれ、鳥の声が埋もれることが多い。

2. 複数種の同時発声（multi-label問題）
    - 一つの音声区間に複数の鳥が同時に鳴くため、単一分類ではなく多ラベル識別が必要。

3. データ不均衡（long-tail distribution）
    - 一部の鳥は大量のデータがある一方、希少種は極端に少なく、モデルが偏りやすい。

4. 弱教師データ（weak labels）
    - 多くのデータでは「録音内に存在する種」は分かるが、正確な発声タイミングは不明。

5. 環境差によるドメインシフト
    - 訓練データと評価データで録音環境や地域が異なり、未知環境への一般化性能が求められる。
---

## 準備

In [1]:
# ライブラリーインポート
#　基本的ライブラリー
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ディスプレイオプション
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set professional plotting style
plt.style.use('ggplot')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.family'] = 'Arial'
custom_palette = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]
sns.set_palette(custom_palette)

In [4]:
# データ読み込み
BASE_DIR = "./data"
# BASE_DIR = "/kaggle/input/Title"
train_df = pl.read_csv(
    BASE_DIR + "/train.csv",
    schema_overrides={"primary_label": pl.String}
)
# 同様に他のファイルも型を明示すると安全です
train_lbl_df = pl.read_csv(BASE_DIR + "/train_soundscapes_labels.csv")
taxonomy_df = pl.read_csv(
    BASE_DIR + "/taxonomy.csv",
    schema_overrides={"primary_label": pl.String}
)

datasets = {
    "train data" : train_df,
    "train label": train_lbl_df,
    "taxonomy data"  :  taxonomy_df,
}

## 簡易的データ観察

### 提供データの構成
- **train.csv** : 学習用データセット 
- **train_soundscapes_labels.csv** : 
- **taxonomy.csv** : 
- **sample_submission.csv** : 提出物のサンプル
---

### train.csvの詳細

In [ ]:
display(train_df.head())

primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
str,str,str,f64,f64,str,str,str,i64,str,str,f64,str,str,str
"""1161364""","""[]""","""[]""",-22.7562,-46.8666,"""Guyalna cuta""","""Guyalna cuta""","""Insecta""",1161364,"""Lucas Barbosa""","""cc-by-nc""",0.0,"""https://static.inaturalist.org…","""1161364/iNat1216197.ogg""","""iNat"""
"""1161364""","""[]""","""[]""",-22.7558,-46.87,"""Guyalna cuta""","""Guyalna cuta""","""Insecta""",1161364,"""Lucas Barbosa""","""cc-by-nc""",0.0,"""https://static.inaturalist.org…","""1161364/iNat1114648.ogg""","""iNat"""
"""1161364""","""[]""","""[]""",-22.7547,-46.8728,"""Guyalna cuta""","""Guyalna cuta""","""Insecta""",1161364,"""Lucas Barbosa""","""cc-by-nc""",0.0,"""https://static.inaturalist.org…","""1161364/iNat810195.ogg""","""iNat"""
"""1161364""","""[]""","""[]""",-22.7547,-46.8728,"""Guyalna cuta""","""Guyalna cuta""","""Insecta""",1161364,"""Lucas Barbosa""","""cc-by-nc""",0.0,"""https://static.inaturalist.org…","""1161364/iNat818781.ogg""","""iNat"""
"""1161364""","""[]""","""[]""",-22.7426,-46.8985,"""Guyalna cuta""","""Guyalna cuta""","""Insecta""",1161364,"""Lucas Barbosa""","""cc-by-nc""",0.0,"""https://static.inaturalist.org…","""1161364/iNat556514.ogg""","""iNat"""


- **primary_label** : 種のコード（鳥類の場合はeBirdコード、鳥類以外の場合はiNaturalistの分類群ID）。すべての種に専用ページがあるわけではありません。リンクが機能しない場合もあります。
- **secondary_labels** : 録音者によって記録にも出現するとマークされた種名のリスト。ほとんどnullです。
- **latitude & longitude** : 録音が行われた場所の座標。鳥の種類によっては、鳴き声に地域的な「方言」がある場合があるため、トレーニングデータに地理的な多様性を持たせることをお勧めします。
- **scientific_name** : 科学的な名前。
- **common_name** : 一般的な名前。
- **class_name** : 生物の分類。
- **author** : 録音を提供したユーザー。
- **license** : 提供データのライセンス。
- **rating** : Xeno-canto のユーザーが提供する 1 ～ 5 の値 (1 - 低品質、5 - 高品質、背景種が存在する場合は評価が 0.5 減少)。0 は評価がないことを意味します。iNaturalist は品質評価を提供していません。
- **url** : 音声データのURL。
- **filename** : 関連付けられた音声ファイルの名前。
- **collection** : Xeno-canto のユーザーが提供する 1 ～ 5 の値 (1 - 低品質、5 - 高品質、背景種が存在する場合は評価が 0.5 減少)。0 は評価がないことを意味します。iNaturalist は品質評価を提供していません。

---

### train_soundscapes_labels.csvの詳細

In [25]:
display(train_lbl_df.head())

filename,start,end,primary_label
str,str,str,str
"""BC2026_Train_0039_S22_20211231…","""00:00:00""","""00:00:05""","""22961;23158;24321;517063;65380"""
"""BC2026_Train_0039_S22_20211231…","""00:00:05""","""00:00:10""","""22961;23158;24321;517063;65380"""
"""BC2026_Train_0039_S22_20211231…","""00:00:10""","""00:00:15""","""22961;23158;24321;517063;65380"""
"""BC2026_Train_0039_S22_20211231…","""00:00:15""","""00:00:20""","""22961;23158;24321;517063;65380"""
"""BC2026_Train_0039_S22_20211231…","""00:00:20""","""00:00:25""","""22961;23158;24321;517063;65380"""


- **filename** : BC2026_Test_ <ファイルID> _ <場所>_ <日にち> _<時間(UTC標準時刻)> .ogg
- **start** : 生物音声の開始点。
- **end** : 生物音声の最後点。
- **primary_label** : 音声データに含まれる生物のID。
---

### taxonmy.csvの詳細

In [22]:
display(taxonomy_df.head())

primary_label,inat_taxon_id,scientific_name,common_name,class_name
str,i64,str,str,str
"""1161364""",1161364,"""Guyalna cuta""","""Guyalna cuta""","""Insecta"""
"""116570""",116570,"""Caiman yacare""","""Southern Spectacled Caiman""","""Reptilia"""
"""1176823""",1176823,"""Leptodactylus luctator""","""Wrestler Frog""","""Amphibia"""
"""1491113""",1491113,"""Adenomera guarani""","""Guaraní leaf-litter frog""","""Amphibia"""
"""1595929""",1595929,"""Lysapsus limellum""","""Uruguay Harlequin Frog""","""Amphibia"""


- **primary_label**: 種のコード(train.csvのlabelと共通)。
- **scientific_name**: 科学的な名前。
- **common_name**: 一般的な名前。
- **class_name**: 生物の分類（鳥類、両生類、哺乳動物、昆虫類、爬虫類）を含む、さまざまな種に関するデータ。
--- 

In [ ]:
# 統計値の確認
for name, df in datasets.items():
    display(f"<{name}>")
    display(df.describe())

'<train data>'

statistic,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
str,str,str,str,f64,f64,str,str,str,f64,str,str,f64,str,str,str
"""count""","""35549""","""35549""","""35549""",35549.0,35549.0,"""35549""","""35549""","""35549""",35549.0,"""35549""","""35549""",35549.0,"""35549""","""35549""","""35549"""
"""null_count""","""0""","""0""","""0""",0.0,0.0,"""0""","""0""","""0""",0.0,"""0""","""0""",0.0,"""0""","""0""","""0"""
"""mean""",null,null,null,-8.166453,-60.7447,null,null,null,80221.321275,null,null,2.600748,null,null,null
"""std""",null,null,null,20.254421,25.434547,null,null,null,242247.605677,null,null,2.070471,null,null,null
"""min""","""1161364""","""['24279']""","""['', ' canto']""",-54.8574,-159.6556,"""Accipiter striatus""","""Amazonian Motmot""","""Amphibia""",7.0,""" Guillermo Treboux""","""by-nc-sa""",0.0,"""https://static.inaturalist.org…","""1161364/iNat1114648.ogg""","""XC"""
"""25%""",null,null,null,-23.3636,-75.1417,null,null,null,8830.0,null,null,0.0,null,null,null
"""50%""",null,null,null,-14.8825,-58.1302,null,null,null,15957.0,null,null,3.5,null,null,null
"""75%""",null,null,null,4.6429,-48.7335,null,null,null,19627.0,null,null,4.5,null,null,null
"""max""","""yeofly1""","""[]""","""[]""",69.578,175.3239,"""Vanellus chilensis""","""Yungas de la Paz Poison Frog""","""Reptilia""",1.595929e6,"""🪶🐾🍀Nandan Sreejith🦌🍄🌍""","""cc0""",5.0,"""https://xeno-canto.org/999798/…","""yeofly1/iNat999817.ogg""","""iNat"""


'<train label>'

statistic,filename,start,end,primary_label
str,str,str,str,str
"""count""","""1478""","""1478""","""1478""","""1478"""
"""null_count""","""0""","""0""","""0""","""0"""
"""mean""",null,null,null,null
"""std""",null,null,null,null
"""min""","""BC2026_Train_0001_S08_20250606…","""00:00:00""","""00:00:05""","""116570;43435;47158son06;47158s…"
"""25%""",null,null,null,null
"""50%""",null,null,null,null
"""75%""",null,null,null,null
"""max""","""BC2026_Train_0066_S23_20241124…","""00:00:55""","""00:01:00""","""thlwre1"""


'<taxonomy data>'

statistic,primary_label,inat_taxon_id,scientific_name,common_name,class_name
str,str,f64,str,str,str
"""count""","""234""",234.0,"""234""","""234""","""234"""
"""null_count""","""0""",0.0,"""0""","""0""","""0"""
"""mean""",null,128502.901709,null,null,null
"""std""",null,307194.585295,null,null,null
"""min""","""1161364""",7.0,"""Accipiter striatus""","""Amazonian Motmot""","""Amphibia"""
"""25%""",null,10915.0,null,null,null
"""50%""",null,19215.0,null,null,null
"""75%""",null,47158.0,null,null,null
"""max""","""yeofly1""",1.595929e6,"""Vanellus chilensis""","""Yungas de la Paz Poison Frog""","""Reptilia"""


## EDA

## データ考察と戦略立案

## モデル構築と予測

今回使用する予測モデルは

## 性能評価
### 評価手法の解説

## 再考察

## コメント

### 最後に
ここまで読んでいただきありがとうございました。私はデータ分析の学習のためにkaggleのコンペティションに参加しています。何かアドバイスや疑問点があればお気軽にコメントしてください。日本語でも英語でもどちらでも対応しています。...

参考

参考文献タイトル: 
[参考文献URL]

参考文献タイトル: 
[参考文献URL]

参考文献タイトル: 
[参考文献URL]